*TEAM NUMBER - 07*

# Instagram Content Recommendation System

This project presents the design and implementation of a data-driven Instagram content recommendation system.The system focuses on generating personalized post recommendations by analyzing user behavior, content attributes, and engagement patterns.

### **Team Members :**
1.  Kruthika S (ENG23DS0015)
2.  Jeevitha A M (ENG23DS0061)
3.  Prabhu Shiva K (ENG23DS0080)

## **Problem Statement :**
Social media platforms produce massive amounts of content, making it hard for users to find what truly interests them.
Single-method recommendation systems often miss the complexity of user behavior and content patterns.
A smarter, multi-approach system is needed to deliver more accurate and relevant recommendations.


## **Methodology Overview:**

To address this problem, the system integrates multiple recommendation techniques:

1. Content-Based Filtering
Recommends posts by analyzing similarities in content features such as caption length, hashtags, and follower count.
2. Collaborative Filtering
Identifies users with similar interaction patterns and recommends posts based on their preferences using cosine similarity.
3. Hybrid Recommendation System
Combines content-based and collaborative filtering to overcome the limitations of individual approaches and improve overall performance.
4. Machine Learning Model (XGBoost Regressor)
Predicts engagement scores of posts using engineered features, enabling the system to prioritize high-performing content.


# **Outputs :**

The system produces the following outputs:

Personalized post recommendations for users
Predicted engagement scores using machine learning
Visualizations including:

1.  Actual vs Predicted comparison
2.  Model performance metrics
3.  Feature importance analysis
4.  Comparison of recommendation techniques





# **Conclusion :**

This project shows that combining multiple recommendation methods with machine learning improves accuracy and reliability. The hybrid model balances content similarity and user behavior, while the predictive model identifies high-engagement posts.


# **Guide :**

Dr. Jobin Thomas

Assistant Professor

Department of CSE (Data Science)

# **Importing Required Libraries**

Contributors: Kruthika S, Jeevitha A M, Prabhu Shiva K

* Imported pandas and numpy for data handling and numerical operations
* Imported cosine similarity for collaborative filtering
* Imported train-test split for model evaluation
* Imported MAE and R² metrics for performance measurement
* Imported XGBoost Regressor for machine learning model development
* Configured matplotlib for generating and saving visualizations

These libraries supported data processing, recommendation logic, model training, and result visualization throughout the system.


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# **Dataset Loading**

Contributor: Kruthika S

Loaded the dataset directly from a GitHub raw URL
Eliminated the need for manual file upload or local file paths
Stored the dataset in a dataframe for further processing
Displayed dataset shape to verify successful loading
Previewed initial rows to understand structure and features

This approach ensured the project is fully reproducible and can be executed seamlessly in any environment.

In [2]:
import pandas as pd


url = "https://raw.githubusercontent.com/kruthikasrinivasa/instagram-recommendation-system/main/Instagram_Analytics.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully from GitHub.")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully from GitHub.
Dataset Shape: (29999, 23)


,post_id,account_id,account_type,follower_count,media_type,content_category,traffic_source,has_call_to_action,post_datetime,post_date,...,comments,shares,saves,reach,impressions,engagement_rate,followers_gained,caption_length,hashtags_count,performance_bucket_label
0,IG0000001,7,brand,3551,reel,Technology,Home Feed,1,2024-11-30 06:00:00,2024-11-30,...,5,7,34,4327,6230,0.0385,899,100,7,medium
1,IG0000002,20,creator,31095,image,Fitness,Hashtags,1,2025-08-15 15:00:00,2025-08-15,...,10,21,68,7451,8268,0.0663,805,122,5,viral
2,IG0000003,15,brand,8167,reel,Beauty,Reels Feed,0,2025-09-11 16:00:00,2025-09-11,...,2,1,22,1639,2616,0.0531,758,115,8,high
3,IG0000004,11,creator,9044,carousel,Music,External,0,2025-09-18 03:00:00,2025-09-18,...,0,7,0,2877,3171,0.0309,402,115,7,medium
4,IG0000005,8,creator,15986,reel,Technology,Profile,0,2025-03-21 09:00:00,2025-03-21,...,8,5,21,5350,8503,0.0221,155,112,9,low


Feature Engineering and Data Transformation

Contributor: Kruthika S

1. Convert categorical engagement labels (low, medium, high, viral) into numerical scores for model compatibility
2. Normalize key numerical features (caption length, hashtags, followers, engagement rate, followers gained) to ensure consistent scale
3. Encode account type (creator, brand) into numerical format
4. Create interaction features to capture relationships between variables
5. Compute a simple content score using weighted contribution of caption, hashtags, and followers

These transformations improve data representation and enhance the performance of recommendation and prediction models.

In [3]:
mapping = {'low': 1, 'medium': 2, 'high': 3, 'viral': 4}
df['engagement_score'] = df['performance_bucket_label'].map(mapping)

df['caption_norm'] = df['caption_length'] / df['caption_length'].max()
df['hashtags_norm'] = df['hashtags_count'] / df['hashtags_count'].max()
df['follower_norm'] = df['follower_count'] / df['follower_count'].max()

df['account_type_encoded'] = df['account_type'].map({'creator': 1, 'brand': 0})

df['engagement_rate_norm'] = df['engagement_rate'] / df['engagement_rate'].max()
df['followers_gained_norm'] = df['followers_gained'] / df['followers_gained'].max()

df['caption_hashtag_interaction'] = df['caption_norm'] * df['hashtags_norm']
df['follower_engagement_interaction'] = df['follower_norm'] * df['engagement_rate_norm']

df['content_score_simple'] = (
    df['caption_norm'] * 0.4 +
    df['hashtags_norm'] * 0.3 +
    df['follower_norm'] * 0.3
)


# **Content-Based Recommendation**

Contributor: Kruthika S

1. Extract user-specific data using the given user ID
2. Create a user profile using average feature values (engagement, caption, hashtags, followers) and most frequent account type
3. Compute similarity between user profile and other posts using distance-based measure
4. Filter out posts already associated with the user
5. Rank posts based on similarity score (lower distance = higher similarity)
6. Select top candidates and return the most relevant recommended posts

This method recommends posts that are similar to the user’s past behavior and content preferences.

In [4]:
def content_based_recommend(user_id, top_n=5):
    user_data = df[df['account_id'] == user_id]

    user_profile = {
        'engagement_score': user_data['engagement_score'].mean(),
        'caption_norm': user_data['caption_norm'].mean(),
        'hashtags_norm': user_data['hashtags_norm'].mean(),
        'account_type_encoded': user_data['account_type_encoded'].mode()[0],
        'follower_norm': user_data['follower_norm'].mean()
    }

    def compute_similarity(row):
        return np.sqrt(
            (row['engagement_score'] - user_profile['engagement_score'])**2 +
            (row['caption_norm'] - user_profile['caption_norm'])**2 +
            (row['hashtags_norm'] - user_profile['hashtags_norm'])**2 +
            (row['account_type_encoded'] - user_profile['account_type_encoded'])**2 +
            (row['follower_norm'] - user_profile['follower_norm'])**2
        )

    df_filtered = df[df['account_id'] != user_id].copy()
    df_filtered['similarity_score'] = df_filtered.apply(compute_similarity, axis=1)

    top_candidates = df_filtered.sort_values(by='similarity_score').head(100)
    rec = top_candidates.sample(n=top_n, random_state=42)
    rec = rec.sort_values(by='similarity_score')

    return rec[['post_id', 'similarity_score']]

# **Collaborative Filtering**

Contributor: Jeevitha A M

1. Constructed a user–item matrix using account IDs, post IDs, and engagement scores
2. Replaced missing values with zero to handle sparse data
3. Computed similarity between users using cosine similarity
4. Identified top similar users for the given user
5. Aggregated their preferences using weighted scoring
6. Filtered out posts already seen by the user
7. Ranked remaining posts and returned top recommendations

This method recommends posts based on the behavior of similar users, enabling discovery beyond direct content similarity.

In [5]:
ratings = df[['account_id', 'post_id', 'engagement_score']]

user_item_matrix = ratings.pivot_table(
    index='account_id',
    columns='post_id',
    values='engagement_score'
).fillna(0)

user_similarity = cosine_similarity(user_item_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

def collaborative_recommend(user_id, top_n=5):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False).drop(user_id)
    top_users = similar_users.head(3)

    similar_posts = user_item_matrix.loc[top_users.index]

    weights = top_users.values
    if weights.sum() == 0:
        weights = np.ones(len(weights))

    scores = similar_posts.T.dot(weights) / (weights.sum() + 1e-8)
    scores = scores.sort_values(ascending=False)

    user_seen = user_item_matrix.loc[user_id]
    unseen = scores[user_seen == 0]

    if unseen.empty:
        unseen = scores

    unseen = unseen.fillna(0)
    unseen += np.random.rand(len(unseen)) * 0.01

    return pd.DataFrame({
        'post_id': unseen.head(top_n).index,
        'collab_score': unseen.head(top_n).values
    })

# **Hybrid Recommendation System**

Contributor: Jeevitha A M

1. Generated recommendations using both content-based and collaborative methods
2. Merged results from both approaches based on post ID
3. Converted similarity scores into comparable content scores
4. Normalized collaborative scores to ensure balanced contribution
5. Combined both scores using weighted averaging
6. Ranked posts based on final hybrid score and selected top recommendations

This approach improved recommendation accuracy by leveraging both user preferences and community behavior.

In [6]:
def hybrid_recommend(user_id, top_n=5):
    content_rec = content_based_recommend(user_id, 20)
    collab_rec = collaborative_recommend(user_id, 20)

    hybrid = pd.merge(content_rec, collab_rec, on='post_id', how='outer').fillna(0)

    hybrid['content_score'] = 1 / (1 + hybrid['similarity_score'])

    if hybrid['collab_score'].max() > 0:
        hybrid['collab_score_norm'] = hybrid['collab_score'] / hybrid['collab_score'].max()
    else:
        hybrid['collab_score_norm'] = 0

    hybrid['final_score'] = 0.5 * hybrid['content_score'] + 0.5 * hybrid['collab_score_norm']

    return hybrid.sort_values(by='final_score', ascending=False).head(top_n)

# **Machine Learning Model Training**

Contributor: Prabhu Shiva K

* Selected relevant engineered features as input variables
* Defined engagement score as the target variable
* Split the dataset into training and testing sets
* Initialized the XGBoost Regressor with tuned parameters
* Trained the model on the training data
* Generated predictions on the test data
* Evaluated performance using MAE and R² score

This step enabled prediction of post engagement levels, helping identify high-performing content for improved recommendations.


# **Machine Learning Model Training**

Contributor: Prabhu Shiva K

Selected relevant engineered features as input variables for the model
Defined engagement score as the target variable to be predicted
Split the dataset into training and testing sets for evaluation
Initialized the XGBoost Regressor with appropriate parameters
Trained the model using the training dataset
Generated predictions on the test dataset
Evaluated model performance using MAE and R² score

This step helped in learning patterns from the data and enabled accurate prediction of post engagement for better recommendation quality.

In [7]:
def train_ml_model():
    X = df[[
        'caption_norm', 'hashtags_norm', 'follower_norm', 'account_type_encoded',
        'engagement_rate_norm', 'followers_gained_norm',
        'caption_hashtag_interaction', 'follower_engagement_interaction',
        'content_score_simple'
    ]]

    y = df['engagement_score']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("\n========== MODEL ==========")
    print("MAE:", round(mean_absolute_error(y_test, y_pred), 3))
    print("R2:", round(r2_score(y_test, y_pred), 3))

    return model, y_test, y_pred

In [8]:
def ml_recommend(user_id, model, top_n=5):
    candidates = df[df['account_id'] != user_id].copy()

    X = candidates[[
        'caption_norm', 'hashtags_norm', 'follower_norm', 'account_type_encoded',
        'engagement_rate_norm', 'followers_gained_norm',
        'caption_hashtag_interaction', 'follower_engagement_interaction',
        'content_score_simple'
    ]]

    candidates['ml_score'] = model.predict(X)

    rec = candidates.sort_values(by='ml_score', ascending=False).head(top_n)

    print("\n ML Recommendations:\n")
    print(rec[['post_id', 'ml_score']].round(3).to_string(index=False))

    return rec

# **Actual vs Predicted Visualization**

Contributor: Prabhu Shiva K

1. Created a scatter plot to compare actual and predicted engagement scores
2. Plotted actual values on the x-axis and predicted values on the y-axis
3. Added labels and title for clear interpretation
4. Adjusted layout for proper visualization
5. Saved the plot as an image file for analysis

This visualization helped in assessing how closely the model predictions match the actual values, indicating the model’s accuracy.

In [9]:
def plot_actual_vs_predicted(y_test, y_pred):
    plt.figure(figsize=(8, 5))
    plt.scatter(y_test, y_pred)
    plt.title("Actual vs Predicted")
    plt.xlabel("Actual Engagement Score")
    plt.ylabel("Predicted Engagement Score")
    plt.tight_layout()
    plt.savefig("actual_vs_predicted.png")
    plt.close()

# **Model Performance Visualization**

Contributor: Prabhu Shiva K

Calculated performance metrics including MAE and R² score
Created a bar chart to visually represent these metrics
Labeled axes and added title for clarity
Annotated each bar with exact metric values
Adjusted layout and saved the chart as an image file

This visualization provided a clear summary of model accuracy and prediction performance.

In [10]:
def plot_model_performance_chart(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    plt.figure(figsize=(6, 4))
    metrics = ['MAE', 'R2 Score']
    values = [mae, r2]

    bars = plt.bar(metrics, values)
    plt.title("Model Performance Metrics")
    plt.ylabel("Value")

    for bar, value in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{value:.3f}",
            ha='center'
        )

    plt.tight_layout()
    plt.savefig("model_performance_chart.png")
    plt.close()

# **Feature Importance Analysis**

Contributor: Prabhu Shiva K , Kruthika S

Defined feature names used in the machine learning model
Extracted feature importance scores from the trained model
Created a structured dataframe for analysis and sorting
Visualized feature importance using a bar chart
Labeled axes and rotated feature names for better readability
Annotated each bar with exact importance values
Saved the visualization as an image file

This analysis helped in identifying which features contributed most to the model’s predictions, improving interpretability of the system.

In [11]:
def plot_feature_importance_clean(model):
    feature_names = [
        'caption_norm', 'hashtags_norm', 'follower_norm', 'account_type_encoded',
        'engagement_rate_norm', 'followers_gained_norm',
        'caption_hashtag_interaction', 'follower_engagement_interaction',
        'content_score_simple'
    ]

    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(10, 5))
    bars = plt.bar(importance_df['Feature'], importance_df['Importance'])
    plt.title("Feature Importance Analysis")
    plt.ylabel("Importance Score")
    plt.xticks(rotation=45, ha='right')

    for bar, value in zip(bars, importance_df['Importance']):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{value:.3f}",
            ha='center',
            fontsize=8
        )

    plt.tight_layout()
    plt.savefig("feature_importance_chart.png")
    plt.close()

    return importance_df

# Recommendation Techniques Comparison

Contributor: Jeevitha A M, Prabhu Shiva K

Generated top recommendations using content-based, collaborative, and hybrid methods
Computed average scores for each method to enable comparison
Predicted engagement scores using the trained ML model and calculated average top scores
Compared all four approaches: Content-Based, Collaborative, Hybrid, and ML-Based
Visualized the comparison using a bar chart
Annotated bars with exact values and saved the chart as an image file

This comparison helped in evaluating the effectiveness of different recommendation techniques and highlighting the improvement achieved through hybrid and ML-based approaches.

In [12]:
def plot_recommendation_comparison(user_id, model):
    content_rec = content_based_recommend(user_id, 5)
    collab_rec = collaborative_recommend(user_id, 5)
    hybrid_rec = hybrid_recommend(user_id, 5)

    content_avg = (1 / (1 + content_rec['similarity_score'])).mean() if not content_rec.empty else 0
    collab_avg = collab_rec['collab_score'].mean() if not collab_rec.empty else 0
    hybrid_avg = hybrid_rec['final_score'].mean() if not hybrid_rec.empty else 0

    candidates = df[df['account_id'] != user_id].copy()
    X_ml = candidates[[
        'caption_norm', 'hashtags_norm', 'follower_norm', 'account_type_encoded',
        'engagement_rate_norm', 'followers_gained_norm',
        'caption_hashtag_interaction', 'follower_engagement_interaction',
        'content_score_simple'
    ]]
    candidates['ml_score'] = model.predict(X_ml)
    ml_avg = candidates.sort_values(by='ml_score', ascending=False).head(5)['ml_score'].mean()

    plt.figure(figsize=(8, 5))
    methods = ['Content-Based', 'Collaborative', 'Hybrid', 'ML-Based']
    values = [content_avg, collab_avg, hybrid_avg, ml_avg]

    bars = plt.bar(methods, values)
    plt.title("Comparison of Recommendation Techniques")
    plt.ylabel("Average Top-5 Score")
    plt.xticks(rotation=15)

    for bar, value in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{value:.3f}",
            ha='center'
        )

    plt.tight_layout()
    plt.savefig("recommendation_comparison_chart.png")
    plt.close()

# **Final Recommendation Table Generation**

Contributor: Jeevitha A M, Prabhu Shiva K

1. Generated top recommendations using the hybrid recommendation method
2. Predicted engagement scores for candidate posts using the trained ML model
3. Merged hybrid scores with ML predictions based on post ID
4. Assigned ranking to recommended posts based on final scores
5. Structured and renamed columns for clear presentation
6. Displayed the final recommendation table in a readable format
7. Visualized the table as an image and saved it for reporting

This step provided a consolidated view of recommendations, combining multiple scores to present a clear and interpretable final output.

In [13]:
def show_clean_recommendation_table(user_id, model, top_n=5):
    hybrid_rec = hybrid_recommend(user_id, top_n).copy()

    candidates = df[df['account_id'] != user_id].copy()
    X_candidates = candidates[[
        'caption_norm', 'hashtags_norm', 'follower_norm', 'account_type_encoded',
        'engagement_rate_norm', 'followers_gained_norm',
        'caption_hashtag_interaction', 'follower_engagement_interaction',
        'content_score_simple'
    ]]
    candidates['ml_score'] = model.predict(X_candidates)

    merged = pd.merge(
        hybrid_rec,
        candidates[['post_id', 'ml_score']],
        on='post_id',
        how='left'
    )

    merged['Rank'] = range(1, len(merged) + 1)

    final_table = merged[['post_id', 'similarity_score', 'collab_score', 'final_score', 'ml_score', 'Rank']].copy()
    final_table.columns = [
        'Post ID',
        'Similarity Score',
        'Collaborative Score',
        'Final Hybrid Score',
        'Predicted Engagement',
        'Rank'
    ]

    print("\n========== CLEAN SAMPLE RECOMMENDATION TABLE ==========\n")
    print(final_table.round(3).to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.axis('off')
    table = ax.table(
        cellText=final_table.round(3).values,
        colLabels=final_table.columns,
        loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)

    plt.title("Sample Recommended Posts Output", pad=20)
    plt.tight_layout()
    plt.savefig("sample_recommendation_table.png", bbox_inches='tight')
    plt.close()

    return final_table


# **Main Execution and System Workflow**

Contributors: Kruthika S, Jeevitha A M, Prabhu Shiva K

Displayed sample user IDs to guide input selection
Accepted user input or assigned a default user ID
Generated recommendations using content-based, collaborative, and hybrid methods
Trained the machine learning model and evaluated its performance
Generated ML-based recommendations for the selected user
Produced multiple visualizations including performance and comparison charts
Generated a final structured recommendation table
Saved all outputs and visualizations as image files

This step executed the complete recommendation pipeline, integrating all components of the system and producing final outputs for analysis and interpretation.

In [14]:
# ==============================
# FINAL POLISHED UI (READABLE + CLEAN)
# ==============================

from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# Train model
model, _, _ = train_ml_model()

# Dropdown
user_dropdown = widgets.Dropdown(
    options=sorted(df['account_id'].unique()),
    description='Select User:',
    layout=widgets.Layout(width='350px')
)

# Button
button = widgets.Button(
    description="Generate Recommendations",
    button_style='success',
    layout=widgets.Layout(width='250px')
)

output = widgets.Output()

# Clean table display (FIXED COLORS)
def display_table(df, title):
    display(HTML(f"<h3 style='color:#00bcd4'>{title}</h3>"))

    styled = df.reset_index(drop=True).style \
        .set_properties(**{
            'background-color': '#ffffff',
            'color': '#000000',
            'border': '1px solid #ccc'
        }) \
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white')]},
            {'selector': 'td', 'props': [('padding', '6px')]}
        ])

    display(styled)

def on_click(b):
    with output:
        clear_output()
        user_id = user_dropdown.value

        display(HTML(f"<h2 style='color:#4CAF50'>Recommendations for User {user_id}</h2>"))

        display_table(content_based_recommend(user_id), "Content-Based Recommendations")
        display_table(collaborative_recommend(user_id), "Collaborative Recommendations")
        display_table(hybrid_recommend(user_id), "Hybrid Recommendations")
        display_table(ml_recommend(user_id, model), "ML-Based Recommendations")

button.on_click(on_click)

display(user_dropdown, button, output)


========== MODEL ==========
MAE: 0.273
R2: 0.726


Dropdown(description='Select User:', layout=Layout(width='350px'), options=(np.int64(1), np.int64(2), np.int64…

Button(button_style='success', description='Generate Recommendations', layout=Layout(width='250px'), style=But…

Output()